# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the [FAIR² Mass Spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library and the Croissant schema.

### Dataset Source
The dataset is described by a Croissant schema publicly available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
List available record sets, their `@id`s, fields, and corresponding IDs for exploration.

We'll also print sample records from each record set for inspection.

In [ ]:
# List all record sets with IDs and their fields
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, dtype: {field.data_type})")
    print()

# Print a few records from each record set (by @id)
for rs in record_sets:
    print(f"Sample records from record set: {rs.name} (@id: {rs.id})")
    for i, rec in enumerate(dataset.records(record_set=rs.id)):
        print(rec)
        if i >= 1:
            break
    print()

## 3. Data Extraction
Load data from each record set into Pandas DataFrames for further analysis. All references use the entity `@id` fields, as required for reproducibility and alignment with Croissant best practices.

In [ ]:
# Collect all record set IDs
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

# Load all records to DataFrames
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# For demonstration, print the columns of the first record set
first_rs_id = record_set_ids[0] if record_set_ids else None
if first_rs_id:
    print(f"Record set {first_rs_id} column names:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Let's select a record set and analyze one of its numeric fields.

We'll choose the first record set and search for a numeric field (typically data type `Float` or `Integer`).

Steps:
- Filter records with the numeric field above a threshold.
- Normalize the field.
- Group by another categorical field (if one exists).

In [ ]:
import numpy as np

# Choose a record set and numeric field
selected_rs = dataset.record_sets[0] if dataset.record_sets else None
if selected_rs is not None:
    df = dataframes[selected_rs.id]
    # Find a numeric field by data_type
    numeric_fields = [f for f in selected_rs.fields if f.data_type in ('Float', 'Integer', 'Number')]
    if numeric_fields:
        numeric_field = numeric_fields[0].id  # Use @id
        print(f"Using numeric field: {numeric_fields[0].name} (@id: {numeric_field})\n")
        # Ensure the field is numeric
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = np.nanmedian(df[numeric_field])
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold} (median): {len(filtered_df)} records\n")

        # Normalize
        m = filtered_df[numeric_field].mean()
        s = filtered_df[numeric_field].std()
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - m) / (s if s else 1)
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field (the first string field not equal to the numeric field)
        cat_fields = [f for f in selected_rs.fields if f.data_type in ('Text', 'String') and f.id != numeric_field]
        group_field = cat_fields[0].id if cat_fields else None
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped data by {group_field} (mean {numeric_field}):\n{grouped_df.head()}")
        else:
            print("No appropriate categorical field to group by.")
    else:
        print("No numeric fields found in the selected record set.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and the result of any grouping (if possible).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_rs is not None and 'filtered_df' in locals() and not filtered_df.empty:
    # Distribution of the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in filtered {selected_rs.name}")
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot by group field (if exists)
    if group_field and group_field in filtered_df.columns:
        plt.figure(figsize=(12, 4))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f"Boxplot of {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion

- We successfully loaded the FAIR² Mass Spectrometry dataset using its Croissant schema and explored its structure using `mlcroissant`.
- The exploration workflow used only the `@id` fields to reference entities for maximum reproducibility and clarity.
- We demonstrated extraction, basic filtering, normalization, and simple visualization of the data.
- For deeper analyses, you can iterate through all available record sets, experiment with different fields, and prepare the data for specific downstream bioinformatics or ML tasks.